# 02 · Audio model v1: predictions on the validation set

This notebook does not train a model. It loads the saved v1 audio model and the
cached Mel-spectrograms from notebook 01, and saves the predictions for the
validation set (true label, predicted label and the six class probabilities).
These predictions were used to measure the calibration of v1 (average
confidence 86% vs. accuracy 58.8%, ECE 0.27), which is one reason we trained
audio model v2.


- The first cell defines the prediction function, but it loaded the first
  checkpoint it found in the `audio_runs` folder. This was the best epoch
  (epoch 17) of an earlier, separate training run, which is not part of this
  project's results.
- The second cell loads the correct v1 checkpoint , which is the model trained in notebook 01. It checks that this
  model reproduces the saved test accuracy (0.6089) and then saves the
  validation predictions.

Only the output of the second cell is used in the report.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

import json, numpy as np, pandas as pd, torch
from pathlib import Path
from torch import nn
from torchvision.models import resnet18


ROOT = Path("/content/drive/MyDrive/Affective Project")
assert ROOT.exists(), f"Not found: {ROOT}"
BASE = next(p for p in ROOT.iterdir() if (p / "audio_features_v1").exists())
FEATURES = BASE / "audio_features_v1"
CKPT = next((BASE / "audio_runs").rglob("best_resnet18.pt"))
print("Folder:", BASE, "\nCheckpoint:", CKPT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ck = torch.load(CKPT, map_location=device, weights_only=True)
classes = ck["classes"]
print("Epoch:", ck["epoch"])

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(ck["model_state_dict"])
model = model.to(device).eval()
mean = torch.tensor(ck["normalization_mean"])[:, None, None]
std = torch.tensor(ck["normalization_std"])[:, None, None]

def predict(split):
    with np.load(FEATURES / f"{split}.npz", allow_pickle=False) as d:
        X, y, fn, spk = d["images"], d["labels"], d["filenames"], d["speakers"]
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), 64):
            x = torch.from_numpy(X[i:i+64]).float() / 255.0
            x = (x.unsqueeze(1).repeat(1, 3, 1, 1) - mean) / std
            probs.append(torch.softmax(model(x.to(device)), 1).cpu().numpy())
    P = np.concatenate(probs); pred = P.argmax(1)
    df = pd.DataFrame({"filename": fn, "Speaker": spk,
                       "true_label": [classes[i] for i in y],
                       "predicted_label": [classes[i] for i in pred],
                       "correct": y == pred})
    for i, c in enumerate(classes):
        df[f"prob_{c}"] = P[:, i]
    return df


test_acc = predict("test")["correct"].mean()
print("Test accuracy:", round(test_acc, 4), "(should be 0.6089)")


val = predict("validation")
print("Validation:", len(val), "samples,", val["Speaker"].nunique(), "speakers, accuracy", round(val["correct"].mean(), 4))
val.to_csv("/content/audio_validation_predictions.csv", index=False)
files.download("/content/audio_validation_predictions.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder: /content/drive/MyDrive/Affective Project/affective computing 
Checkpoint: /content/drive/MyDrive/Affective Project/affective computing/audio_runs/20261008_135720/best_resnet18.pt
Epoch: 17
Test accuracy: 0.6108 (should be 0.6089)
Validation: 1222 samples, 15 speakers, accuracy 0.5933


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
CKPT = BASE / "audio_runs" / "20261014_143230" / "best_resnet18.pt"
ck = torch.load(CKPT, map_location=device, weights_only=True)
print("Epoch:", ck["epoch"])
assert ck["epoch"] == 13, "Still the wrong checkpoint"
model.load_state_dict(ck["model_state_dict"])
model.eval()

test_acc = predict("test")["correct"].mean()
print("Test accuracy:", round(test_acc, 4), "(should be 0.6089)")
assert abs(test_acc - 0.6089) < 0.001

val = predict("validation")
print("Validation:", len(val), "samples, accuracy", round(val["correct"].mean(), 4))
val.to_csv("/content/audio_validation_predictions.csv", index=False)
files.download("/content/audio_validation_predictions.csv")

Epoch: 13
Test accuracy: 0.6089 (should be 0.6089)
Validation: 1222 samples, accuracy 0.5884


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>